# 03 — Training

Fine-tune encoder models and run LLM baselines.  
All runs use the same hyperparameters — the only variable is which density column is used for weighting.

**Sections**
1. Configuration
2. Define experiments (dataset × density column)
3. Fine-tuning runs
4. LLM baselines
5. Results summary

In [1]:
import sys, os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import json
import yaml
import pandas as pd

from src.training import TrainingConfig, train, run_baselines

c:\Users\Alexandre\miniconda3\envs\faiss2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

Edit `TrainingConfig` fields below to change model, hyperparameters, or output paths.  
All experiments in this notebook share the same config — only `density_column` varies per run.

In [2]:
CONFIG_PATH = "configs/datasets.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

EMBEDDINGS_ROOT   = cfg["embedding"]["output_root"]
PREPROCESSED_ROOT = cfg["preprocessing"]["output_root"]
MODEL_NAME        = cfg["embedding"]["models"][0]
MODEL_SLUG        = MODEL_NAME.replace("/", "_")
K_VALUES          = cfg["embedding"]["k_values"]

# Russian is reference-only: not used for fine-tuning, only for density + eval
TRAIN_DATASETS    = cfg.get("train_datasets", ["toxigen"])

train_cfg = cfg["training"]

base_config = TrainingConfig(
    model_id     = train_cfg["models"][0],
    batch_size   = train_cfg["batch_size"],
    learning_rate= train_cfg["learning_rate"],
    num_epochs   = train_cfg["epochs"],
    max_length   = train_cfg["max_length"],
    random_state = train_cfg["random_state"],
    output_root  = train_cfg["output_root"],
)

print("Models         :", train_cfg["models"])
print("Epochs         :", base_config.num_epochs)
print("LR             :", base_config.learning_rate)
print("Train datasets :", TRAIN_DATASETS)
print("Output         :", base_config.output_root)

Models         : ['answerdotai/ModernBERT-base', 'tomh/toxigen_roberta']
Epochs         : 3
LR             : 2e-5
Train datasets : ['toxigen']
Output         : outputs/3_training


## 2. Define Experiments

Each experiment is `(dataset, density_column)`.  
- `density_column = None` → train without weighting (baseline encoder)
- `density_column = 'density_k5_ratio'` → weight by Russian/All ratio at K=5 on raw embeddings
- `density_column = 'density_pca_k5_ratio'` → same but PCA space

Edit the list below to add/remove experiments.

In [3]:
# Build experiment list: one fine-tune per (train_dataset × density_column)
# Russian is excluded — it is the reference group used for density, not for training.
density_columns = [None]  # baseline: no weighting
for k in K_VALUES:
    density_columns.append(f"density_k{k}_ratio")       # raw space
    density_columns.append(f"density_pca_k{k}_ratio")   # PCA space

experiments = [
    {"dataset": ds, "density_column": dc, "model_id": mid}
    for ds in TRAIN_DATASETS
    for dc in density_columns
    for mid in train_cfg["models"]
]

print(f"{len(experiments)} fine-tunes planned:")
for ex in experiments:
    mid_slug = ex['model_id'].replace('/', '_')
    tag = f"{mid_slug}__{ex['dataset']}__{ex['density_column'] or 'no_density'}"
    print(f"  {tag}")

14 fine-tunes planned:
  answerdotai_ModernBERT-base__toxigen__no_density
  tomh_toxigen_roberta__toxigen__no_density
  answerdotai_ModernBERT-base__toxigen__density_k5_ratio
  tomh_toxigen_roberta__toxigen__density_k5_ratio
  answerdotai_ModernBERT-base__toxigen__density_pca_k5_ratio
  tomh_toxigen_roberta__toxigen__density_pca_k5_ratio
  answerdotai_ModernBERT-base__toxigen__density_k100_ratio
  tomh_toxigen_roberta__toxigen__density_k100_ratio
  answerdotai_ModernBERT-base__toxigen__density_pca_k100_ratio
  tomh_toxigen_roberta__toxigen__density_pca_k100_ratio
  answerdotai_ModernBERT-base__toxigen__density_k1000_ratio
  tomh_toxigen_roberta__toxigen__density_k1000_ratio
  answerdotai_ModernBERT-base__toxigen__density_pca_k1000_ratio
  tomh_toxigen_roberta__toxigen__density_pca_k1000_ratio


## 3. Fine-Tuning Runs

Each run loads `outputs/2_embeddings/{dataset}/{model_slug}/densities.csv`,  
fine-tunes the model, and saves the checkpoint + `metrics.json` to `outputs/3_training/`.

In [4]:
all_metrics = {}

for ex in experiments:
    ds = ex["dataset"]
    dc = ex["density_column"]
    mid = ex["model_id"]
    mid_slug = mid.replace('/', '_')
    dataset_tag = ds

    density_csv = os.path.join(EMBEDDINGS_ROOT, ds, MODEL_SLUG, "densities.csv")
    if not os.path.exists(density_csv):
        print(f"[SKIP] {density_csv} not found")
        continue

    # Clone config and set density column for this run
    from dataclasses import replace
    run_config = TrainingConfig(
        model_id      = mid,
        batch_size    = base_config.batch_size,
        learning_rate = base_config.learning_rate,
        num_epochs    = base_config.num_epochs,
        max_length    = base_config.max_length,
        random_state  = base_config.random_state,
        output_root   = base_config.output_root,
        density_column= dc,
    )

    run_name = f"{mid_slug}__{run_config.run_name(dataset_tag)}"
    metrics_path = os.path.join(run_config.output_dir(dataset_tag), "metrics.json")

    if os.path.exists(metrics_path):
        print(f"[CACHE] {run_name} — loading existing metrics")
        with open(metrics_path) as f:
            all_metrics[run_name] = json.load(f)
        continue

    print(f"\n{'='*60}")
    print(f"Training: {run_name}")
    print(f"{'='*60}")
    metrics = train(density_csv=density_csv, dataset_tag=dataset_tag, config=run_config)
    all_metrics[run_name] = metrics

print("\nAll fine-tuning runs complete.")

[CACHE] answerdotai_ModernBERT-base__toxigen__no_density — loading existing metrics

Training: tomh_toxigen_roberta__toxigen__no_density


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 31001.72it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
2000,0.207908,0.135038,0.943973,0.925544,0.897361,0.886957,0.892128,0.986001
4000,0.229732,0.245370,0.917135,0.909057,0.809916,0.892143,0.849044,0.972511
6000,0.262626,0.202009,0.922255,0.894402,0.862120,0.836079,0.848900,0.973201
8000,0.271763,0.231445,0.920442,0.882622,0.881496,0.803432,0.840656,0.970592
10000,0.252139,0.226296,0.920163,0.889707,0.862583,0.825934,0.843861,0.971925
12000,0.243384,0.219800,0.923391,0.887527,0.884855,0.812433,0.847099,0.973674
14000,0.195551,0.219703,0.926798,0.903590,0.863426,0.854996,0.859191,0.976196
16000,0.201577,0.245610,0.928751,0.903309,0.873765,0.850038,0.861738,0.967234
18000,0.174841,0.246376,0.924726,0.909288,0.841531,0.876964,0.858882,0.969339
20000,0.205262,0.231858,0.923590,0.909876,0.835346,0.881159,0.857641,0.974167


Writing model shards: 100%|██████████| 1/1 [00:16<00:00, 16.59s/it]


Training Loss,Validation Loss,Step,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
0.140106,0.135038,37644,0.943973,0.925544,0.897361,0.886957,0.892128,0.986001


...

Step,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
2000,0.433835,0.467214,0.895582,0.923795,0.654821,0.969722,0.781751,0.976411
4000,0.534640,0.302787,0.896833,0.833643,0.733326,0.730778,0.732050,0.940320
6000,0.666255,0.601780,0.873954,0.700338,0.854162,0.417716,0.561056,0.905603
8000,0.447893,0.557566,0.868772,0.758277,0.690808,0.578408,0.629631,0.860553
10000,0.570516,0.621694,0.864040,0.684255,0.802113,0.391593,0.526263,0.805876
12000,0.484063,0.670104,0.844099,0.605260,0.896892,0.216466,0.348759,0.726892
14000,0.538552,0.674635,0.823932,0.544313,0.976726,0.089133,0.163358,0.718037
16000,0.690632,0.794238,0.807153,0.500000,0.000000,0.000000,0.000000,0.296601
18000,0.709681,0.754152,0.807153,0.500000,0.000000,0.000000,0.000000,0.289314
20000,0.558377,0.599557,0.807153,0.500000,0.000000,0.000000,0.000000,0.396846


Writing model shards: 100%|██████████| 1/1 [00:16<00:00, 16.05s/it]


Training Loss,Validation Loss,Step,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
0.748401,0.467214,37644,0.895582,0.923795,0.654821,0.969722,0.781751,0.976411


Writing model shards: 100%|██████████| 1/1 [00:16<00:00, 16.57s/it]



All fine-tuning runs complete.


## 4. LLM Baselines

Requires Ollama running locally (`ollama serve`).  
Runs zero-shot, context, and few-shot classification on the Russian test split.

In [5]:
RUN_BASELINES = False  # set True to run (requires Ollama)

if RUN_BASELINES:
    # Use the Russian annotated test set
    russian_test_csv = os.path.join(PREPROCESSED_ROOT, "russian", "test.csv")
    if os.path.exists(russian_test_csv):
        baseline_results = run_baselines(
            test_csv=russian_test_csv,
            dataset_tag="russian_annotated",
            config=base_config,
        )
        print("Baseline results:")
        for mode, metrics in baseline_results.items():
            print(f"  {mode}: F1={metrics['f1']:.4f}  Acc={metrics['accuracy']:.4f}")
    else:
        print(f"Russian test CSV not found at {russian_test_csv}")
else:
    print("Baselines skipped (RUN_BASELINES=False).")

Baselines skipped (RUN_BASELINES=False).


## 5. Results Summary

In [6]:
if all_metrics:
    rows = []
    for run_name, m in all_metrics.items():
        parts = run_name.split("__", 2)
        rows.append({
            "run": run_name,
            "model": parts[0],
            "dataset": parts[1] if len(parts) > 1 else "n/a",
            "density": parts[2] if len(parts) > 2 else "n/a",
            "f1":               m.get("f1", float("nan")),
            "accuracy":         m.get("accuracy", float("nan")),
            "balanced_accuracy": m.get("balanced_accuracy", float("nan")),
            "auc_roc":          m.get("auc_roc", float("nan")),
        })

    summary = pd.DataFrame(rows).sort_values("f1", ascending=False)
    display(summary.style.format({
        "f1": "{:.4f}",
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "auc_roc": "{:.4f}",
    }))
else:
    print("No metrics collected yet — run the training cells first.")

,run,model,dataset,density,f1,accuracy,balanced_accuracy,auc_roc
1,tomh_toxigen_roberta__toxigen__no_density,tomh_toxigen_roberta,toxigen,no_density,0.8921,0.9440,0.9255,0.9860
0,answerdotai_ModernBERT-base__toxigen__no_density,answerdotai_ModernBERT-base,toxigen,no_density,0.8692,0.9310,0.9136,0.9795
5,tomh_toxigen_roberta__toxigen__density_pca_k5_ratio,tomh_toxigen_roberta,toxigen,density_pca_k5_ratio,0.7999,0.9117,0.9081,0.9708
9,tomh_toxigen_roberta__toxigen__density_pca_k100_ratio,tomh_toxigen_roberta,toxigen,density_pca_k100_ratio,0.7944,0.9146,0.8944,0.9661
12,answerdotai_ModernBERT-base__toxigen__density_pca_k1000_ratio,answerdotai_ModernBERT-base,toxigen,density_pca_k1000_ratio,0.7926,0.9163,0.8831,0.9553
8,answerdotai_ModernBERT-base__toxigen__density_pca_k100_ratio,answerdotai_ModernBERT-base,toxigen,density_pca_k100_ratio,0.7908,0.9215,0.8656,0.9544
13,tomh_toxigen_roberta__toxigen__density_pca_k1000_ratio,tomh_toxigen_roberta,toxigen,density_pca_k1000_ratio,0.7818,0.8956,0.9238,0.9764
4,answerdotai_ModernBERT-base__toxigen__density_pca_k5_ratio,answerdotai_ModernBERT-base,toxigen,density_pca_k5_ratio,0.7712,0.9165,0.8420,0.9541
3,tomh_toxigen_roberta__toxigen__density_k5_ratio,tomh_toxigen_roberta,toxigen,density_k5_ratio,0.7677,0.9219,0.8248,0.9699
2,answerdotai_ModernBERT-base__toxigen__density_k5_ratio,answerdotai_ModernBERT-base,toxigen,density_k5_ratio,0.7561,0.9135,0.8296,0.9460
